In [ ]:
pip install pandas numpy textblob vaderSentiment scikit-learn


In [21]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as vader
from textblob import TextBlob as tb
import pickle

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print('ok')

ok


In [9]:
df = pd.read_csv('IMDB_Dataset.csv')
df.head()
print(df)

                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


In [6]:
# VADER POLARITY ANALYSIS (TEXT TO NUMBER STEP)
def vader_textblob(text):

    sid = vader()
    vad = sid.polarity_scores(text)
    blob = tb(text).sentiment


    return pd.Series({
        'vader_neg': vad['neg'],
        'vader_neu': vad['neu'],
        'vader_pos': vad['pos'],
        'vader_compound': vad['compound'],
        'textblob_polarity': blob.polarity,
        'textblob_subjectivity': blob.subjectivity
    })
X = df['review'].apply(vader_textblob)
y = df['sentiment'].map({'positive': 1, 'negative': 0})


In [9]:
print(X)
# print(y.head())

       vader_neg  vader_neu  vader_pos  vader_compound  textblob_polarity  \
0          0.179      0.756      0.064         -0.9916           0.023433   
1          0.052      0.773      0.176          0.9670           0.109722   
2          0.114      0.688      0.198          0.9519           0.354008   
3          0.125      0.816      0.059         -0.9213          -0.057813   
4          0.050      0.806      0.144          0.9744           0.217952   
...          ...        ...        ...             ...                ...   
49995      0.045      0.765      0.189          0.9886           0.394425   
49996      0.160      0.730      0.110         -0.6837          -0.276190   
49997      0.181      0.704      0.115         -0.9734           0.056984   
49998      0.116      0.804      0.080         -0.8657          -0.048663   
49999      0.118      0.734      0.148          0.6975           0.120000   

       textblob_subjectivity  
0                   0.490369  
1            

In [13]:
# print(X.head())
print(y)

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int64


In [25]:
#FUNCTIONS DECLARATION

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# X: (n_features, m), Y: (1, m), W: (n_features, 1)
def gradient_descent(W, b, X, Y, epochs, learning_rate):
    m = X.shape[1]
    for i in range(epochs):
        # FORWARD PROPAGATION
        Z = np.dot(W.T, X) + b
        A = sigmoid(Z)

        # LOSS FUNCTION (binary cross-entropy)
        cost = -(1 / m) * np.sum(Y * np.log(A + 1e-8) + (1 - Y) * np.log(1 - A + 1e-8))

        # BACKWARD PROPAGATION
        dZ = A - Y
        dW = (1 / m) * np.dot(X, dZ.T)
        db = (1 / m) * np.sum(dZ)

        # UPDATE
        W -= learning_rate * dW
        b -= learning_rate * db

        if i % 100 == 0 or i == epochs - 1:
            print(f"Epoch {i:04d} | Loss: {cost:.4f}")

    return W, b



In [ ]:
# MLP TRAINING (sklearn)

# TRAIN TEST SPLIT & SCALING
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# TRAIN MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    max_iter=500,
    random_state=42,
    verbose=True
)
mlp.fit(X_train_scaled, y_train)

y_pred = mlp.predict(X_test_scaled)


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

cwd = os.getcwd()
results_dir = os.path.join(cwd, 'results')
eval_dir = os.path.join(results_dir, 'evaluation_results')
os.makedirs(eval_dir, exist_ok=True)

# Confusion matrix image
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4, 4))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha='center', va='center', color='black')
plt.tight_layout()
plt.savefig(os.path.join(eval_dir, 'confusion_matrix.png'), dpi=150)
plt.close()

# Classification report image
report = classification_report(y_test, y_pred, output_dict=True)
labels = ['0', '1', 'accuracy', 'macro avg', 'weighted avg']
metrics = ['precision', 'recall', 'f1-score']

report_matrix = []
row_labels = []
for lbl in labels:
    if lbl == 'accuracy':
        continue
    if lbl in report:
        row = [report[lbl].get(m, np.nan) for m in metrics]
        report_matrix.append(row)
        row_labels.append(lbl)

if 'accuracy' in report:
    report_matrix.append([np.nan, np.nan, report['accuracy']])
    row_labels.append('accuracy')

report_matrix = np.array(report_matrix, dtype=np.float32)

plt.figure(figsize=(6, 3.5))
plt.imshow(report_matrix, cmap='Greens', aspect='auto')
plt.title('Classification Report')
plt.xticks(range(len(metrics)), metrics)
plt.yticks(range(len(row_labels)), row_labels)
for (i, j), v in np.ndenumerate(report_matrix):
    if np.isfinite(v):
        plt.text(j, i, f"{v:.2f}", ha='center', va='center', color='black')
plt.tight_layout()
plt.savefig(os.path.join(eval_dir, 'classification_report.png'), dpi=150)
plt.close()

print(f"Saved evaluation images to {eval_dir}")

In [ ]:
# Save trained model

cwd = os.getcwd()
results_dir = os.path.join(cwd, 'results')
os.makedirs(results_dir, exist_ok=True)

model_artifact = {
    'model': mlp,
    'scaler': scaler,
    'feature_columns': list(X.columns)
}

model_path = os.path.join(results_dir, 'amine_assignment_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model_artifact, f)

print(f"Saved model to {model_path}")